In [172]:
import pandas
import pathlib

In [173]:
files = list(pathlib.Path('.').glob('data/*.csv'))

In [174]:
df = pandas.concat([pandas.read_csv(f) for f in files], ignore_index=True)

In [175]:
df.head()

,title,name,age,height,elected_in,place_of_residence,result,ethnicity,hair_color,year,country,competition
0,Miss Albigeois,Karine Lherm,19,171,Lisle-sur-Tarn,Cestayrols,NaN,W,D,2000,France,Miss France
1,Miss Alsace,Barbara Fink,20,178,Brumath,Brumath,SF,W,D,2000,France,Miss France
2,Miss Anjou,Laureen Bidi,19,180,Vern-d'Anjou,Cholet,NaN,W,D,2000,France,Miss France
3,Miss Aquitaine,Sonia Benlloch,19,177,Castillon-la-Bataille,Port-Sainte-Mari,NaN,W,D,2000,France,Miss France
4,Miss Artois-Hainaut,Sophie Andry,20,180,Valenciennes,Béthune,NaN,W,D,2000,France,Miss France


## Geocode the cities

In [176]:
fr_post_codes_path = pathlib.Path('..').joinpath('geo/fr_code_postal_v2.csv').absolute()

In [177]:
fr_post_codes = pandas.read_csv(fr_post_codes_path)

In [178]:
fr_post_codes = fr_post_codes[['commune', 'gps']]

In [179]:
fr_post_codes.commune = fr_post_codes.commune.apply(lambda x: x.lower())

In [180]:
french_pageants = df[df.country.str.contains('rance', na=False)]
french_pageants.country.count()

np.int64(43)

In [181]:
for pageant in french_pageants.itertuples(name='Pageant'):
    municipality = fr_post_codes[fr_post_codes.commune == pageant.elected_in.lower()]
    df.loc[pageant.Index, 'municipality'] = municipality.commune.values[-1] if not municipality.empty else None
    df.loc[pageant.Index, 'municipality_gps'] = municipality.gps.values[-1] if not municipality.empty else None

## Geocode countries

- ICU
- Languages

## Additional infos

In [182]:
df['birth_year'] = df['year'] - df['age']

In [183]:
df['elected_in_wikidata'] = pandas.NA

In [184]:
df.to_csv('pageants.csv', index=False)

In [185]:
elected_in_cities = df[~df.elected_in.duplicated() & df.elected_in_wikidata.isna()][['elected_in', 'elected_in_wikidata']]

In [186]:
elected_in_cities.sort_values('elected_in', inplace=True)

In [187]:
elected_in_cities.to_csv('elected_in_cities_to_geocode.csv', index=False)